In [2]:
import os
import re

vault_path = r"C:\Users\ASUS\Videos\AnyDesk\Balasubramanian PG\Projects\DP 700 Professional Certification"
master_file = os.path.join(vault_path, "Master.md")

def extract_title_from_line(line):
    match = re.search(r"\[\[(.*?)\]\]", line)
    return match.group(1) if match else None

def clean_filename(name):
    return re.sub(r'[<>:"/\\|?*]', '', name)

with open(master_file, "r", encoding="utf-8") as f:
    for line in f:
        title_path = extract_title_from_line(line)
        if title_path:
            parts = title_path.split('/')
            folder = os.path.join(vault_path, *parts[:-1]) if len(parts) > 1 else vault_path
            os.makedirs(folder, exist_ok=True)
            filename = clean_filename(parts[-1]) + ".md"
            filepath = os.path.join(folder, filename)
            if not os.path.exists(filepath):
                with open(filepath, "w", encoding="utf-8") as nf:
                    nf.write(f"# {parts[-1]}\n\nAuto-generated note.\n")
                print(f"Created: {filepath}")


Created: C:\Users\ASUS\Videos\AnyDesk\Balasubramanian PG\Projects\DP 700 Professional Certification\Ingest data with Microsoft Fabric.md
Created: C:\Users\ASUS\Videos\AnyDesk\Balasubramanian PG\Projects\DP 700 Professional Certification\Ingest Data with Dataflows Gen2 in Microsoft Fabric.md
Created: C:\Users\ASUS\Videos\AnyDesk\Balasubramanian PG\Projects\DP 700 Professional Certification\Introduction.md
Created: C:\Users\ASUS\Videos\AnyDesk\Balasubramanian PG\Projects\DP 700 Professional Certification\Understand Dataflows Gen2 in Microsoft Fabric.md
Created: C:\Users\ASUS\Videos\AnyDesk\Balasubramanian PG\Projects\DP 700 Professional Certification\Explore Dataflows Gen2 in Microsoft Fabric.md
Created: C:\Users\ASUS\Videos\AnyDesk\Balasubramanian PG\Projects\DP 700 Professional Certification\Integrate Dataflows Gen2 and Pipelines in Microsoft Fabric.md
Created: C:\Users\ASUS\Videos\AnyDesk\Balasubramanian PG\Projects\DP 700 Professional Certification\Exercise - Create and use a Dataflo

In [ ]:
import os
import re

vault_path = r"C:\Users\ASUS\Videos\AnyDesk\Balasubramanian PG\Data Lemur"
master_file = os.path.join(vault_path, "Data Lemur.md")

def extract_title_from_line(line):
    match = re.search(r"\[\[(.*?)\]\]", line)
    return match.group(1) if match else None

def clean_filename(name):
    return re.sub(r'[<>:"/\\|?*]', '', name)

with open(master_file, "r", encoding="utf-8") as f:
    for line in f:
        title_path = extract_title_from_line(line)
        if title_path:
            parts = title_path.split('/')
            folder = os.path.join(vault_path, *parts[:-1]) if len(parts) > 1 else vault_path
            os.makedirs(folder, exist_ok=True)
            filename = clean_filename(parts[-1]) + ".md"
            filepath = os.path.join(folder, filename)
            if not os.path.exists(filepath):
                with open(filepath, "w", encoding="utf-8") as nf:
                    nf.write(f"# {parts[-1]}\n\nAuto-generated note.\n")
                print(f"Created: {filepath}")


In [1]:
import os
import re
import shutil

# --- CONFIGURATION ---
# 1. Set the absolute path to your Obsidian vault.
#    Use r"..." to handle backslashes correctly on Windows.
vault_path = r"C:\Users\ASUS\Videos\AnyDesk\Balasubramanian PG\Projects\DP 700 Professional Certification"

# 2. Set the name of your master markdown file.
master_md_file_name = "Master.md"
# --- END CONFIGURATION ---

master_md_path = os.path.join(vault_path, master_md_file_name)

# Regex to find the wikilink [[...]] within a table cell.
note_link_pattern = re.compile(r'\[\[(.*?)\]\]')

# Check if the master file exists before proceeding.
if not os.path.exists(master_md_path):
    print(f"ERROR: Master file not found at '{master_md_path}'")
    exit()

print(f"Starting organization process for vault: '{vault_path}'")
print("-" * 30)

# Open and read the master markdown file.
with open(master_md_path, 'r', encoding='utf-8') as f:
    # Skip the first two lines of a markdown table (header and separator)
    next(f, None) 
    next(f, None)

    for line in f:
        # A valid markdown table row starts with '|'
        if not line.strip().startswith('|'):
            continue

        # Split the row into cells based on the '|' delimiter.
        cells = [cell.strip() for cell in line.strip().split('|')]

        # Ensure the row has enough columns to parse
        if len(cells) < 7:
            continue

        # --- 1. Parse Data from Table Cells ---
        # The category is in the 2nd column (index 1)
        # The sub-category is in the 3rd column (index 2)
        # The title is in the 7th column (index 6)
        
        # Clean up the category/sub-category names (remove bolding **)
        category = cells[1].replace('**', '').strip()
        sub_category = cells[2].replace('**', '').strip()
        title_cell = cells[6]

        # Skip if any essential part is empty
        if not category or not sub_category or not title_cell:
            continue
            
        # Find the note path inside the [[...]]
        match = note_link_pattern.search(title_cell)
        if not match:
            continue
            
        note_path = match.group(1)

        # --- 2. Construct File Paths ---
        # Get the base filename (e.g., "My Note.md")
        source_note_name = os.path.basename(note_path) + ".md"
        
        # The original location of the note file (in the vault's root)
        source_file_path = os.path.join(vault_path, source_note_name)

        # The new target folder based on Category and Sub-Category
        target_folder_path = os.path.join(vault_path, category, sub_category)

        # The final destination path for the note file
        target_file_path = os.path.join(target_folder_path, source_note_name)

        # --- 3. Perform File Operations ---
        # Check if the source file actually exists before trying to move it
        if os.path.exists(source_file_path):
            # Create the nested target folders if they don't exist
            os.makedirs(target_folder_path, exist_ok=True)
            
            # Move the file from the source to the target destination
            shutil.move(source_file_path, target_file_path)
            
            # Use relpath to make the output cleaner
            relative_target = os.path.relpath(target_folder_path, vault_path)
            print(f"Moved: '{source_note_name}'  ->  '{relative_target}'")
        else:
            # If the file is not in the root, it might have been moved already.
            # You can add a check here to see if it's already in its target location.
            if not os.path.exists(target_file_path):
                print(f"SKIPPED: Source file '{source_note_name}' not found in the root directory.")

print("-" * 30)
print("Organization complete.")

Starting organization process for vault: 'C:\Users\ASUS\Videos\AnyDesk\Balasubramanian PG\Projects\DP 700 Professional Certification'
------------------------------
------------------------------
Organization complete.
